In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from auction_module.bundling_and_bidding.fitness_functions.vrp_learn.distance import (
    my_convex_hull_jaccard_distance,
    my_hausdorff_distance,
    my_modified_hausdorff_distance,
)
import numpy as np


## Plotting and evaluation

In [ ]:
df = pd.read_parquet('data/experiments/new_experiments/new_experiments_2026-05-26T20:07:37.parquet')

,meta__start,meta__run_id,meta__base_seed,meta__trial_index,meta__status,meta__end,params__x_min,params__x_max,params__y_min,params__y_max,...,params__pred_num_locations,params__opt_algorithm,params__maxeval,params__seed,res__xopt,res__opt_val,res__return_code,res__trajectory,res__proxy_objectives,res__true_objectives
0,2026-05-26 17:22:00.978911,afd760b1eebed718,1,0,success,2026-05-26 17:22:32.876411,0,100,0,100,...,4,2,256,12670702393961142041,"[[16.666666666666668, 22.016460905349795], [33...",15.990231,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[121.75051334594035, 95.83644922470782, 92.162...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
1,2026-05-26 17:22:32.877391,afd760b1eebed718,1,1,success,2026-05-26 17:23:04.348742,0,100,0,100,...,4,2,256,12670702393961142042,"[[19.958847736625515, 99.38271604938271], [32....",52.620695,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[173.01571460419427, 145.16327014778912, 122.6...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
2,2026-05-26 17:23:04.349620,afd760b1eebed718,1,2,success,2026-05-26 17:23:39.453854,0,100,0,100,...,4,2,256,12670702393961142043,"[[44.78737997256516, 22.016460905349795], [80....",39.669730,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[130.24544137895958, 91.03364762547967, 103.93...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
3,2026-05-26 17:23:39.454840,afd760b1eebed718,1,3,success,2026-05-26 17:24:13.978427,0,100,0,100,...,4,2,256,12670702393961142044,"[[92.38683127572017, 70.98765432098763], [33.1...",38.742741,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[149.55141256437534, 113.67854898792471, 104.7...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
4,2026-05-26 17:24:13.979308,afd760b1eebed718,1,4,success,2026-05-26 17:24:42.910983,0,100,0,100,...,4,2,256,12670702393961142045,"[[33.95061728395062, 53.703703703703695], [42....",33.467335,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[65.95216069242917, 58.36576479409826, 48.8505...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2026-05-26 20:02:32.987142,afd760b1eebed718,1,195,success,2026-05-26 20:03:40.057149,0,100,0,100,...,4,2,256,12670702393961142236,"[[74.69135802469134, 22.839506172839506], [33....",28.997845,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[125.2327832478381, 94.70645437350086, 89.7176...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
196,2026-05-26 20:03:40.058454,afd760b1eebed718,1,196,success,2026-05-26 20:04:40.399438,0,100,0,100,...,4,2,256,12670702393961142237,"[[83.33333333333333, 81.27572016460906], [89.5...",14.667140,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[96.82361540450759, 69.24277940695333, 68.4657...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
197,2026-05-26 20:04:40.400793,afd760b1eebed718,1,197,success,2026-05-26 20:05:41.974505,0,100,0,100,...,4,2,256,12670702393961142238,"[[52.469135802469125, 73.45679012345678], [47....",32.633380,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[114.45932683709091, 71.54849753838302, 89.160...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."
198,2026-05-26 20:05:41.976152,afd760b1eebed718,1,198,success,2026-05-26 20:06:37.927661,0,100,0,100,...,4,2,256,12670702393961142239,"[[50.0, 46.2962962962963], [50.0, 16.666666666...",36.424580,5,"[[[50.0, 50.0], [50.0, 50.0], [50.0, 50.0], [5...","[66.75983073675367, 47.10294576775427, 53.9067...","[{'my_convex_hull_jaccard_distance': 1.0, 'my_..."


In [ ]:
# create some additional metrics for plotting
plot_obj_name = 'my_hausdorff_distance'
proxy = res['proxy_objectives']
true_obj = [x[plot_obj_name] for x in res['true_objectives']]
# minimum proxy so far
proxy_running_min = list(itertools.accumulate(proxy, min))
# hausdorff at the minimum proxy
# Zip both lists, then track the running minimum based on proxy
running_min_with_hausdorff = itertools.accumulate(
    zip(proxy, true_obj), 
    lambda current, next_item: next_item if next_item[0] < current[0] else current
)
# Extract only the corresponding 'foo' values
hausdorff_following_min = [item[1] for item in running_min_with_hausdorff]

# (proxy and) hausdorff of n random samples: the no-optmization benchmark
num_samples = 1000
records = []
rng = np.random.default_rng(1)
for i in range(num_samples):
    true_base_locations = rng.uniform((x_min, y_min), (x_max, y_max), size=(true_num_locations, 2))
    pred_base_locations = rng.uniform((x_min, y_min), (x_max, y_max), size=(pred_num_locations, 2))
    # df = pd.DataFrame(np.concat([set_a, set_b], axis=0), columns=['x', 'y'])
    record = {
        'idx': i,
        'true_base_locations':true_base_locations,
        'pred_base_locations': pred_base_locations,
        'my_hausdorff_distance': my_hausdorff_distance(true_base_locations, pred_base_locations),
        'my_modified_hausdorff_distance': my_modified_hausdorff_distance(true_base_locations, pred_base_locations),
        'my_convex_hull_jaccard_distance': my_convex_hull_jaccard_distance(true_base_locations, pred_base_locations),
    }
    record['normalized hausdorff'] = record['my_hausdorff_distance']/np.sqrt((x_max-x_min)**2 + (y_max - y_min)**2)
    record['normalized modified hausdorff'] = record['my_modified_hausdorff_distance']/np.sqrt((x_max-x_min)**2 + (y_max - y_min)**2)
    records.append(record)
no_opt_df = pd.DataFrame.from_records(records)



In [ ]:
plt.plot(proxy, 'b', label='Proxy', markersize=3, alpha=0.3)
plt.plot(proxy_running_min, 'b--', label='proxy running min', markersize=3, )
plt.plot(true_obj, 'r', label=plot_obj_name, markersize=3, alpha=0.3)
plt.plot(hausdorff_following_min, 'r--', label=f'{plot_obj_name} following min proxy',)
plt.boxplot(no_opt_df[plot_obj_name], positions=[len(proxy)], widths=10, label=f'no-opt {plot_obj_name} (boxplot)')
plt.hlines(no_opt_df[plot_obj_name].mean(), 0, len(proxy), linestyles='-.', label=f'no-opt {plot_obj_name} (mean)')
# plt.yscale('log')
plt.legend()

In [ ]:
# plot the trajectory of points
foo = np.array(res["trajectory"])
t, n, d = foo.shape
for i in range(n):
    x = foo[:, i, 0]
    y = foo[:, i, 1]
    plt.plot(
        x, # + rng.normal(0, 2, x.shape),  # jitter for e.g. DIRECT
        y, #+ rng.normal(0, 2, y.shape),
        "-o", 
        alpha=0.3
    )
    plt.xlim(0, 100)
    plt.ylim(0, 100)
    if i > 4:
        break

In [ ]:
plt.plot(res['trajectory'][0][:, 0], 
res['trajectory'][0][:, 1], 'o')

In [ ]:
res['trajectory'][-1][:, 0], res['trajectory'][-1][:, 1]

## Scratches

In [ ]:
from functools import cache
from time import sleep

@cache
def bar_internal(x: np.array):
    sleep(2)
    return sum(x)

def bar(x):
    normalized_input=tuple(sorted(x))
    return bar_internal(np.array(normalized_input))

In [ ]:
bar((1, 2, 3, 4, 5))

In [ ]:
bar((5, 4, 3, 2, 5))

In [ ]:
bar((5, 4, 3, 2, 1))

In [ ]:
doo = np.array([0, 0, 3, 4, 3, 1, 1, 2])
doo = doo.reshape(-1, 2, copy=True)

In [ ]:
doo

In [ ]:
ind = np.lexsort([doo[:, 1], doo[:, 0]])
doo[ind]

In [ ]:
goo = np.array([0, 0, 3, 4, 3, 1, 1, 2])
structured_goo = goo.view(dtype=[('f0', goo.dtype), ('f1', goo.dtype)])
sorted_goo = np.sort(structured_goo, order=['f0', 'f1'])
sorted_goo

In [ ]:
sorted(((0, 0), (3, 4), (3, 1), (1, 2)))

In [ ]:
from new_experiments import solve_tsp_pyvrp
from pyvrp.stop import MaxRuntime
rng2 = np.random.default_rng(2)
locations = rng2.uniform(0, 100, size=(20, 2))
min_cost  = solve_tsp_pyvrp(locations, stop=MaxRuntime(10))
print(min_cost)
